In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

In [12]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [2]:
DATA_PATH = "../data/processed/loan_default_clean.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (255347, 17)


,Age,Income,LoanAmount,CreditScore,MonthsEmployed,NumCreditLines,InterestRate,LoanTerm,DTIRatio,Education,EmploymentType,MaritalStatus,HasMortgage,HasDependents,LoanPurpose,HasCoSigner,Default
0,56,85994,50587,520,80,4,15.23,36,0.44,Bachelor's,Full-time,Divorced,Yes,Yes,Other,Yes,0
1,69,50432,124440,458,15,1,4.81,60,0.68,Master's,Full-time,Married,No,No,Other,Yes,0
2,46,84208,129188,451,26,3,21.17,24,0.31,Master's,Unemployed,Divorced,Yes,Yes,Auto,No,1
3,32,31713,44799,743,0,3,7.07,24,0.23,High School,Full-time,Married,No,No,Business,No,0
4,60,20437,9139,633,8,4,6.51,48,0.73,Bachelor's,Unemployed,Divorced,No,Yes,Auto,No,0


In [3]:
TARGET = "Default"

X = df.drop(columns=[TARGET])
y = df[TARGET]

In [4]:
numeric_features = [
    "Age",
    "Income",
    "LoanAmount",
    "CreditScore",
    "MonthsEmployed",
    "NumCreditLines",
    "InterestRate",
    "LoanTerm",
    "DTIRatio"
]

categorical_features = [
    "Education",
    "EmploymentType",
    "MaritalStatus",
    "HasMortgage",
    "HasDependents",
    "LoanPurpose",
    "HasCoSigner"
]

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 204277
Testing samples: 51070


In [6]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            "passthrough",
            numeric_features
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="first"
            ),
            categorical_features
        )
    ]
)

# Logistic Regression

In [13]:
logistic_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            numeric_features
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="first"
            ),
            categorical_features
        )
    ]
)

In [15]:
logistic_pipeline = Pipeline([
    ("preprocessor", logistic_preprocessor),
    (
        "model",
        LogisticRegression(
            max_iter=2000,
            class_weight="balanced"
        )
    )
])

In [16]:
print("Training Logistic Regression...")

logistic_pipeline.fit(X_train, y_train)

print("Logistic Regression training complete.")

Training Logistic Regression...
Logistic Regression training complete.


# Decision Tree

In [25]:
decision_tree_pipeline = Pipeline([
    (
        "preprocessor",
        preprocessor
    ),
    (
        "model",
        DecisionTreeClassifier(
            class_weight="balanced",
            random_state=42
        )
    )
])

In [26]:
print("Training Decision Tree...")

decision_tree_pipeline.fit(X_train, y_train)

print("Decision Tree training complete.")

Training Decision Tree...
Decision Tree training complete.


# Random Forest

In [27]:
random_forest_pipeline = Pipeline([
    (
        "preprocessor",
        preprocessor
    ),
    (
        "model",
        RandomForestClassifier(
            n_estimators=300,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        )
    )
])

In [28]:
print("Training Random Forest...")

random_forest_pipeline.fit(X_train, y_train)

print("Random Forest training complete.")

Training Random Forest...
Random Forest training complete.


In [33]:
logistic_predictions = logistic_pipeline.predict(X_test)

decision_tree_predictions = decision_tree_pipeline.predict(X_test)

random_forest_predictions = random_forest_pipeline.predict(X_test)

In [34]:
logistic_probabilities = logistic_pipeline.predict_proba(X_test)[:, 1]

decision_tree_probabilities = decision_tree_pipeline.predict_proba(X_test)[:, 1]

random_forest_probabilities = random_forest_pipeline.predict_proba(X_test)[:, 1]

In [35]:
comparison = pd.DataFrame({
    "Actual": y_test.values,
    "Logistic Regression": logistic_predictions,
    "Decision Tree": decision_tree_predictions,
    "Random Forest": random_forest_predictions
})

comparison.head(20)

,Actual,Logistic Regression,Decision Tree,Random Forest
0,0,0,0,0
1,0,0,0,0
2,0,0,0,0
3,0,1,0,0
4,0,0,0,0
5,0,0,0,0
6,0,1,0,0
7,0,0,0,0
8,1,1,0,0
9,0,1,0,0


In [37]:
# Save the trained models

joblib.dump(
    logistic_pipeline,
    "../models/logistic_regression_pipeline.pkl"
)

joblib.dump(
    decision_tree_pipeline,
    "../models/decision_tree_pipeline.pkl"
)

joblib.dump(
    random_forest_pipeline,
    "../models/random_forest_pipeline.pkl"
)

print("All trained models saved successfully.")

All trained models saved successfully.
